### Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import seaborn as sns
from itertools import product
from collections import Counter, defaultdict
import warnings

from sympy import false

warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

### Helper class

In [3]:
class BooleanNetwork:
    def __init__(self, node_names):
        self.nodes = {name: 0 for name in node_names}
        self.rules = {}
        self.history = []
        self.graph = nx.DiGraph()  # NetworkX graph for visualization

        # Add nodes to NetworkX graph
        self.graph.add_nodes_from(node_names)

    def add_rule(self, target_node, rule_function, rule_description=""):
        """
        Add Boolean rule for a node

        Args:
            target_node: Node to update
            rule_function: Function that takes current state dict and returns True/False
            rule_description: Human-readable description
        """
        self.rules[target_node] = {
            'function': rule_function,
            'description': rule_description
        }

    def set_state(self, **kwargs):
        """Set states of specific nodes"""
        for node, value in kwargs.items():
            if node in self.nodes:
                self.nodes[node] = int(bool(value))

    def get_state_vector(self):
        """Get current state as list in sorted order"""
        return [self.nodes[node] for node in sorted(self.nodes.keys())]

    def update_synchronous(self):
        """Update all nodes simultaneously"""
        new_state = {}
        for node in self.nodes:
            if node in self.rules:
                new_state[node] = int(self.rules[node]['function'](self.nodes))
            else:
                new_state[node] = self.nodes[node]  # No rule = no change

        self.nodes = new_state
        self.history.append(self.get_state_vector())

    def simulate(self, steps=10, record_history=True):
        """Run simulation"""
        if record_history:
            self.history = [self.get_state_vector()]

        for step in range(steps):
            self.update_synchronous()

            # Check for steady state
            if len(self.history) >= 2 and self.history[-1] == self.history[-2]:
                print(f"   Reached steady state after {step+1} steps")
                break

        return np.array(self.history)

### Version 1 regulatory network:

In [10]:
## Unmutated network (from practical)

network = BooleanNetwork(nodes)

network.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network.add_rule('p21', lambda s: s['p53'], "p21 = p53")
network.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']),
                        "MYC = (NOT p53) AND (NOT p21)")
network.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']),
                        "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
network.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'],
                        "p53 = DNA_damage AND (NOT MDM2)")
network.add_rule('Growth', lambda s: s['CDK2'] and s['MYC'] and (not s['p53']),
                        "Growth = CDK2 AND MYC AND (NOT p53)")
network.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']),
                        "Death = p53 AND DNA_damage AND (NOT Growth)")

In [4]:
# Mutation A

nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network_A = BooleanNetwork(nodes)

# Define Boolean rules (based on real biology, simplified)
network_A.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network_A.add_rule('p21', lambda s: s['p53'], "p21 = p53")
network_A.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
network_A.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network_A.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
network_A.add_rule('p53', lambda s: False, "p53 = BROKEN (always OFF)")
network_A.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
network_A.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")


print("Rules:")
for node, rule_info in network_A.rules.items():
    print(f" {rule_info['description']}")

Rules:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = BROKEN (always OFF)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)


In [5]:
# Mutation B
nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network_B = BooleanNetwork(nodes)

# Define Boolean rules (based on real biology, simplified)
network_B.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network_B.add_rule('p21', lambda s: s['p53'], "p21 = p53")
network_B.add_rule('MYC', lambda s: True, "MYC = AMPLIFIED (always ON)")
network_B.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network_B.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
network_B.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'], "p53 = DNA_damage AND (NOT MDM2)")
network_B.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
network_B.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

print("Rules:")
for node, rule_info in network_B.rules.items():
    print(f" {rule_info['description']}")

Rules:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = AMPLIFIED (always ON)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)


In [6]:
# mutation C
nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network_C = BooleanNetwork(nodes)

# Define Boolean rules (based on real biology, simplified)
network_C.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network_C.add_rule('p21', lambda s: s['p53'], "p21 = p53")
network_C.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
network_C.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network_C.add_rule('MDM2', lambda s: True, "MDM2 = OVEREXPRESSED (always ON)")
network_C.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'], "p53 = DNA_damage AND (NOT MDM2)")
network_C.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
network_C.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

print("Rules:")
for node, rule_info in network_C.rules.items():
    print(f" {rule_info['description']}")

Rules:
 DNA_damage = INPUT (constant)
 p21 = p53
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = OVEREXPRESSED (always ON)
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)


In [9]:
# Mutation D - knockout p21 (CDK2 depends on p53 and is overreactive)
nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network_D = BooleanNetwork(nodes)

# Define Boolean rules (based on real biology, simplified)
network_D.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
network_D.add_rule('p21', lambda s: False, "p21 is knocked out (always OFF)")
network_D.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
network_D.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
network_D.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
network_D.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'], "p53 = DNA_damage AND (NOT MDM2)")
network_D.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
network_D.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")

print("Rules:")
for node, rule_info in network_D.rules.items():
    print(f" {rule_info['description']}")

Rules:
 DNA_damage = INPUT (constant)
 p21 is knocked out (always OFF)
 MYC = (NOT p53) AND (NOT p21)
 CDK2 = MYC AND (NOT p21) AND (NOT p53)
 MDM2 = MYC
 p53 = DNA_damage AND (NOT MDM2)
 Growth = CDK2 AND MYC AND (NOT p53)
 Death = p53 AND DNA_damage AND (NOT Growth)


### Scenario Analysis

In [13]:
# 3 scenarios to test
scenarios = {
    'Healthy Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },

    'Stressed Cell': {
        'DNA_damage': 1, 'p53': 0, 'MYC': 0, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    },

    'Oncogene Hijacked Cell': {
        'DNA_damage': 0, 'p53': 0, 'MYC': 1, 'CDK2': 0,
        'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0
    }
}

networks = {
    'Normal': network,
    'Mutation A - p53 Knockout': network_A,
    'Mutation B - MYC Amplification': network_B,
    'Mutation C - MDM2 Overexpression': network_C,
    'Mutation D - p21 Knockout': network_D
}


scenario_results = {}

for network_name, current_network in networks.items():

    print("\n" + "=" * 60)
    print(network_name)
    print("=" * 60)

    scenario_results[network_name] = {}

    for scenario_name, initial_state in scenarios.items():

        # Initial state
        current_network.set_state(**initial_state)

        trajectory = current_network.simulate(steps=8)

        scenario_results[network_name][scenario_name] = trajectory

        # convert into dictionary
        node_names = sorted(current_network.nodes.keys())
        final_state = trajectory[-1]
        final_dict = {
            node: final_state[i]
            for i, node in enumerate(node_names)
        }

        print(
            f"{scenario_name}: "
            f"Growth={final_dict['Growth']}, "
            f"Death={final_dict['Death']}, "
            f"p53={final_dict['p53']}"
        )


Normal
   Reached steady state after 4 steps
Healthy Cell: Growth=1, Death=0, p53=0
   Reached steady state after 6 steps
Stressed Cell: Growth=0, Death=1, p53=1
   Reached steady state after 3 steps
Oncogene Hijacked Cell: Growth=1, Death=0, p53=0

Mutation A - p53 Knockout
   Reached steady state after 4 steps
Healthy Cell: Growth=1, Death=0, p53=0
   Reached steady state after 4 steps
Stressed Cell: Growth=1, Death=0, p53=0
   Reached steady state after 3 steps
Oncogene Hijacked Cell: Growth=1, Death=0, p53=0

Mutation B - MYC Amplification
   Reached steady state after 4 steps
Healthy Cell: Growth=1, Death=0, p53=0
   Reached steady state after 7 steps
Stressed Cell: Growth=1, Death=0, p53=0
   Reached steady state after 3 steps
Oncogene Hijacked Cell: Growth=1, Death=0, p53=0

Mutation C - MDM2 Overexpression
   Reached steady state after 4 steps
Healthy Cell: Growth=1, Death=0, p53=0
   Reached steady state after 7 steps
Stressed Cell: Growth=1, Death=0, p53=0
   Reached steady 

In [ ]:
def canonical_attractor(cycle):
    ## Repeating cycle detection
    cycle = [tuple(state) for state in cycle]

    rotations = [
        tuple(cycle[i:] + cycle[:i])
        for i in range(len(cycle))
    ]

    return min(rotations)


def find_attractor(network, initial_state, max_steps=50):
    
    ## Simulates future from initialstate
    node_names = sorted(network.nodes.keys())

    ## init. state
    network.set_state(**initial_state)

    seen_states = {}
    trajectory = []

    for step in range(max_steps):

        current_state = tuple(network.get_state_vector())

        if current_state in seen_states:

            start_index = seen_states[current_state]
            cycle = trajectory[start_index:]

            return canonical_attractor(cycle)

        seen_states[current_state] = len(trajectory)
        trajectory.append(current_state)

        network.update_synchronous()

    return None

In [15]:
def is_cancerous(attractor, node_names):
    ##Defining cancerous attractor as a cell in which:
    ## DNA damage is present, growth continues, and cell death is off (growth itself is not necessarily cancerous)

    for state in attractor:

        state_dict = {
            node_names[i]: state[i]
            for i in range(len(node_names))
        }

        if not (
            state_dict['DNA_damage'] == 1
            and state_dict['Growth'] == 1
            and state_dict['Death'] == 0
        ):
            return False

    return True

### Attractor Analysis

In [16]:
attractor_results = {}
summary_results = []


for network_name, current_network in networks.items():

    node_names = sorted(current_network.nodes.keys())
    n_nodes = len(node_names)

    # 2^8 possible initial states
    all_states = list(product([0, 1], repeat=n_nodes))

    basin_data = defaultdict(list)

    # Test  all initial state
    for initial_state in all_states:

        state_dict = {
            node_names[i]: initial_state[i]
            for i in range(n_nodes)
        }

        attractor = find_attractor(
            current_network,
            state_dict
        )

        if attractor is not None:
            basin_data[attractor].append(initial_state)


    # Sort from largest to smallest
    attractors = sorted(
        basin_data.keys(),
        key=lambda x: len(basin_data[x]),
        reverse=True
    )

    print("\n" + "=" * 70)
    print(network_name)
    print("=" * 70)

    print(f"Total initial states tested: {len(all_states)}")
    print(f"Number of attractors found: {len(attractors)}")

    cancerous_states = 0


    for i, attractor in enumerate(attractors):

        basin_size = len(basin_data[attractor])

        basin_percentage = (
            basin_size / len(all_states)
        ) * 100

        cancerous = is_cancerous(
            attractor,
            node_names
        )

        if cancerous:
            cancerous_states += basin_size


        # Fixed point or limit cycle?
        if len(attractor) == 1:
            attractor_type = "Fixed point"
        else:
            attractor_type = f"Limit cycle ({len(attractor)} states)"


        print(f"\nAttractor {i + 1}")
        print(f"Type: {attractor_type}")

        print(
            f"Basin size: {basin_size} states "
            f"({basin_percentage:.1f}%)"
        )

        print(
            f"Cancerous: "
            f"{'YES' if cancerous else 'NO'}"
        )


        #biological nodes
        for state_number, state in enumerate(attractor):

            state_dict = {
                node_names[j]: state[j]
                for j in range(len(node_names))
            }

            print(
                f"  State {state_number + 1}: "
                f"DNA_damage={state_dict['DNA_damage']}, "
                f"Growth={state_dict['Growth']}, "
                f"Death={state_dict['Death']}, "
                f"p53={state_dict['p53']}, "
                f"MYC={state_dict['MYC']}"
            )


    # % of all initial states that reach cancer-like attractors
    cancer_percentage = (
        cancerous_states / len(all_states)
    ) * 100


    attractor_results[network_name] = {
        'attractors': attractors,
        'basins': basin_data,
        'cancer_percentage': cancer_percentage
    }

    summary_results.append({
        'Network': network_name,
        'No. of attractors': len(attractors),
        'Cancerous states': cancerous_states,
        'Cancerous percentage': cancer_percentage
    })


Normal
Total initial states tested: 256
Number of attractors found: 3

Attractor 1
Type: Fixed point
Basin size: 128 states (50.0%)
Cancerous: NO
  State 1: DNA_damage=0, Growth=1, Death=0, p53=0, MYC=1

Attractor 2
Type: Fixed point
Basin size: 120 states (46.9%)
Cancerous: NO
  State 1: DNA_damage=1, Growth=0, Death=1, p53=1, MYC=0

Attractor 3
Type: Fixed point
Basin size: 8 states (3.1%)
Cancerous: YES
  State 1: DNA_damage=1, Growth=1, Death=0, p53=0, MYC=1

Mutation A - p53 Knockout
Total initial states tested: 256
Number of attractors found: 2

Attractor 1
Type: Fixed point
Basin size: 128 states (50.0%)
Cancerous: NO
  State 1: DNA_damage=0, Growth=1, Death=0, p53=0, MYC=1

Attractor 2
Type: Fixed point
Basin size: 128 states (50.0%)
Cancerous: YES
  State 1: DNA_damage=1, Growth=1, Death=0, p53=0, MYC=1

Mutation B - MYC Amplification
Total initial states tested: 256
Number of attractors found: 2

Attractor 1
Type: Fixed point
Basin size: 128 states (50.0%)
Cancerous: NO
  St

In [17]:
summary_df = pd.DataFrame(summary_results)

summary_df

,Network,No. of attractors,Cancerous states,Cancerous percentage
0,Normal,3,8,3.125
1,Mutation A - p53 Knockout,2,128,50.000
2,Mutation B - MYC Amplification,2,128,50.000
3,Mutation C - MDM2 Overexpression,2,128,50.000
4,Mutation D - p21 Knockout,5,8,3.125
